<center><img src="./image/CLBLOGO.jpg" alt="创乐博" style="width: 300px;"/></center>

# 7.红外遥控器控制

@－－－－湖南创乐博智能科技有限公司－－－－<br>
@  文件名：7.红外遥控器控制.py <br>
@  版本：V2.0 <br>
@  author: zhulin<br>
@  说明：红外遥控器控制
红外遥控器控制程序，通过pylirc库接收红外遥控器信号，把接收到的红外遥控器控制信号解码后，判断哪个键按下，来控制机器人运动！！！<br>

## 1.导入必要的库文件

In [1]:
import multiprocessing
import time
import sys
from LOBOROBOT import LOBOROBOT
from gpiozero import Button, LED
import pylirc as lirc

## 2. 定义机器人对象

In [2]:
# =====================================================================
# 进程1：红外遥控接收进程
# 专门负责监听遥控器信号，通过队列传递给主进程，避免阻塞主循环
# =====================================================================
def ir_receive_process(cmd_queue):
    # 初始化 pylirc
    lirc.init("pylirc", "/etc/lirc/conf", 0)
    print("[系统日志] 红外遥控接收进程已独立启动...")
    
    while True:
        try:
            s = lirc.nextcode(1)
            while s:
                for code in s:
                    cmd = code["config"]
                    print(f"\n[红外信号] 接收到遥控器指令: {cmd}")
                    # 将指令放入队列，供主进程提取
                    cmd_queue.put(cmd)
                
                # 继续读取缓冲区
                if not 0: # 对应原代码 makerobo_blocking = 0
                    s = lirc.nextcode(1)
                else:
                    s = []
            time.sleep(0.05) # 短暂休眠，防止CPU占用过高
        except Exception as e:
            print(f"[红外进程异常] {e}")
            time.sleep(1)

## 3.初始化函数

In [ ]:
# =====================================================================
# 主函数：避障逻辑与运动执行
# =====================================================================
if __name__ == '__main__':
    # 1. 创建多进程通信队列
    cmd_queue = multiprocessing.Queue()
    
    # 2. 启动红外接收子进程
    ir_process = multiprocessing.Process(target=ir_receive_process, args=(cmd_queue,))
    ir_process.daemon = True # 设为守护进程，主进程退出时一并退出
    ir_process.start()

    # 3. 硬件初始化 (必须在主进程初始化，避免多进程抢占GPIO资源)
    print("[系统日志] 正在初始化机器人硬件...")
    clbrobot = LOBOROBOT() 
    SensorRight = Button(16, pull_up=True)  # 右侧红外避障传感器
    SensorLeft  = Button(12, pull_up=True)  # 左侧红外避障传感器
    Btn         = Button(19, pull_up=True)  # 启动/停止按键
    Gpin        = LED(5)                    # 绿色LED
    Rpin        = LED(6)                    # 红色LED

    # 系统状态变量
    system_running = False
    current_cmd = None           # 当前正在执行的遥控指令
    cmd_remaining_time = 0.0     # 当前指令还需执行的时间
    last_loop_time = time.time() # 上一次循环的时间戳
    last_action_printed = None   # 用于记录上一次打印的动作，避免重复刷屏

    # 按键中断回调函数
    def system_start():
        global system_running
        print('\n*****************************************') 
        print('* 系统启动: 避障与红外控制已激活! *') 
        print('*****************************************')
        Rpin.on()
        Gpin.off()
        system_running = True

    def system_stop():
        global system_running
        print('\n[系统日志] 按键已释放，系统挂起')
        Rpin.off()
        Gpin.on()
        system_running = False
        clbrobot.t_stop(0)

    # 绑定实体按键事件
    Btn.when_pressed = system_start
    Btn.when_released = system_stop

    print("[系统日志] 初始化完成。请按下启动按键 (Button 19) 开始运行。")

    try:
        while True:
            current_time = time.time()
            delta_time = current_time - last_loop_time
            last_loop_time = current_time

            # --------------------------------------------------
            # 步骤 A：从队列获取最新的遥控指令
            # --------------------------------------------------
            while not cmd_queue.empty():
                new_cmd = cmd_queue.get()
                current_cmd = new_cmd
                
                # 如果是停止按键，立刻清零时间并停车
                if current_cmd == 'KEY_NEXT':
                    cmd_remaining_time = 0.0
                    clbrobot.t_stop(0)
                    print("[动作执行] 遥控器请求停止")
                else:
                    # 根据不同的指令分配不同的执行时间
                    if current_cmd in ['KEY_CHANNELUP', 'KEY_CHANNELDOWN']:
                        cmd_remaining_time = 0.3
                    else:
                        cmd_remaining_time = 3.0
                    print(f"[动作规划] 准备执行指令: {current_cmd}，分配时间: 3秒")

            # --------------------------------------------------
            # 步骤 B：系统运行时的核心业务逻辑（避障优先）
            # --------------------------------------------------
            if system_running:
                SR_val = SensorRight.value
                SL_val = SensorLeft.value

                # 判断是否有障碍物 (原代码逻辑: 0表示无障碍，1表示有障碍)
                is_obstacle = not (SL_val == 0 and SR_val == 0)

                if is_obstacle:
                    # 发现障碍物，进入避障模式
                    if last_action_printed != "OBSTACLE":
                        print(f"\n[避障系统] ⚠️ 发现障碍物！ 方位 -> 左侧: {'有' if SL_val==1 else '无'} | 右侧: {'有' if SR_val==1 else '无'}")
                        last_action_printed = "OBSTACLE"

                    if SL_val == 0 and SR_val == 1:     
                        print("           -> 动作: 障碍物在右侧，向左转")
                        clbrobot.turnLeft(50, 0)
                    elif SL_val == 1 and SR_val == 0:   
                        print("           -> 动作: 障碍物在左侧，向右转")
                        clbrobot.turnRight(50, 0)
                    else:
                        print("           -> 动作: 正前方有障碍，后退并左转")
                        clbrobot.t_stop(0.3)
                        clbrobot.t_down(50, 0.4)
                        clbrobot.turnLeft(50, 0.5)
                        
                    # 避障过程中消耗的时间，不计入遥控指令的执行时间
                    # 避障结束后，重置 last_loop_time 相当于“冻结”了之前的任务计时
                    last_loop_time = time.time() 

                else:
                    # 没有障碍物，安全环境，执行或恢复红外遥控的指令
                    if current_cmd and cmd_remaining_time > 0:
                        cmd_remaining_time -= delta_time # 扣减运行时间
                        
                        # 只有当刚从避障状态恢复，或者指令切换时，才打印并下发电机指令
                        if last_action_printed != current_cmd:
                            print(f"[动作恢复/执行] 正在执行: {current_cmd}，剩余时间: {cmd_remaining_time:.1f}秒")
                            
                            if current_cmd == 'KEY_CHANNELDOWN':
                                clbrobot.forward_Left(50, 0.5)
                            elif current_cmd == 'KEY_CHANNEL':
                                clbrobot.t_up(50, 0)
                            elif current_cmd == 'KEY_CHANNELUP':
                                clbrobot.forward_Right(50, 0.5)
                            elif current_cmd == 'KEY_PREVIOUS':
                                clbrobot.moveLeft(50, 0)
                            elif current_cmd == 'KEY_PLAYPAUSE':
                                clbrobot.moveRight(50, 0)
                            elif current_cmd == 'KEY_VOLUMEDOWN':
                                clbrobot.backward_Left(50, 0)
                            elif current_cmd == 'KEY_VOLUMEUP':
                                clbrobot.t_down(50, 0)
                            elif current_cmd == 'KEY_EQUAL':
                                clbrobot.backward_Right(50, 0)

                            last_action_printed = current_cmd

                        # 检查时间是否耗尽
                        if cmd_remaining_time <= 0:
                            print(f"[动作完成] 指令 {current_cmd} 的 3 秒执行完毕，停车。")
                            clbrobot.t_stop(0)
                            current_cmd = None
                            last_action_printed = None
                    
                    elif not current_cmd:
                        # 既没有障碍物，也没有正在执行的遥控任务，保持停止
                        if last_action_printed != "STOP":
                            clbrobot.t_stop(0)
                            last_action_printed = "STOP"

            time.sleep(0.01) # 循环延时，避免CPU跑到100%

    except KeyboardInterrupt:
        print("\n[系统日志] 收到退出信号 (Ctrl+C)，正在释放资源...")
        clbrobot.t_stop(0)
        ir_process.terminate() # 强制结束红外子进程
        sys.exit()

[系统日志] 红外遥控接收进程已独立启动...
[系统日志] 正在初始化机器人硬件...
[系统日志] 初始化完成。请按下启动按键 (Button 19) 开始运行。

[系统日志] 按键已释放，系统挂起

*****************************************
* 系统启动: 避障与红外控制已激活! *
*****************************************

[红外信号] 接收到遥控器指令: KEY_CHANNELUP
[动作规划] 准备执行指令: KEY_CHANNELUP，分配时间: 3秒
[动作恢复/执行] 正在执行: KEY_CHANNELUP，剩余时间: 0.3秒
[动作完成] 指令 KEY_CHANNELUP 的 3 秒执行完毕，停车。

[红外信号] 接收到遥控器指令: KEY_CHANNELDOWN
[动作规划] 准备执行指令: KEY_CHANNELDOWN，分配时间: 3秒
[动作恢复/执行] 正在执行: KEY_CHANNELDOWN，剩余时间: 0.3秒
[动作完成] 指令 KEY_CHANNELDOWN 的 3 秒执行完毕，停车。

[红外信号] 接收到遥控器指令: KEY_CHANNEL
[动作规划] 准备执行指令: KEY_CHANNEL，分配时间: 3秒
[动作恢复/执行] 正在执行: KEY_CHANNEL，剩余时间: 3.0秒

[避障系统] ⚠️ 发现障碍物！ 方位 -> 左侧: 无 | 右侧: 有
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在右侧，向左转
           -> 动作: 障碍物在

## 4 .红外遥控控制函数

## 7.循环函数

## 8.释放函数

## 9.程序入口